# Deploy to an online endpoint

To consume a model from an application, you can deploy the model to an online endpoint. You'll create an MLflow model from local files and test the endpoint.

## Before you start

You'll need the latest version of the  **azure-ai-ml** package to run the code in this notebook. Run the cell below to verify that it is installed.

> **Note**:
> If the **azure-ai-ml** package is not installed, run `pip install azure-ai-ml` to install it.

In [ ]:
pip show azure-ai-ml

## Connect to your workspace

With the required SDK packages installed, now you're ready to connect to your workspace.

To connect to a workspace, we need identifier parameters - a subscription ID, resource group name, and workspace name. Since you're working with a compute instance, managed by Azure Machine Learning, you can use the default values to connect to the workspace.

In [1]:
import os
# Here we set the working directory to the project root to ensure imports work correctly
from pathlib import Path
target = "dp100-learn"
p = Path.cwd()
print(f"Starting working directory: {p}")
while p.name != target and p.parent != p:
    p = p.parent
# Set the path to your project root manually if the above code does not work
p = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn"
# p = "C:/Users/dmika/DEV/Projects-local/dp100-learn"
os.chdir(p)
print("Changed working directory to:", p)
from utils.azureml_utils import *

# Get Azure ML Client based on your environment. Learn more in the tutorials/azureml-first-notebook.ipynb.
ml_client = get_azureml_client()

Starting working directory: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code
Changed working directory to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn
Added to sys.path: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn


Found the config file in: /config.json


Added to sys.path: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn


In [ ]:
import mlflow

azure_labs_path = "azure-labs/azure-ml-dev/Labs/11"
os.chdir(azure_labs_path)

In [ ]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

try:
    credential = DefaultAzureCredential()
    # Check if given credential can get token successfully.
    credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    # Fall back to InteractiveBrowserCredential in case DefaultAzureCredential not work
    credential = InteractiveBrowserCredential()

In [ ]:
# Get a handle to workspace
ml_client = MLClient.from_config(credential=credential)

## Define and create an endpoint

Ultimately, the goal is to deploy a model to an endpoint. Therefore, you first need to create an endpoint. The endpoint will be a HTTPS endpoint that an application can call to receive predictions from the model. An application can consume an endpoint by using its URI, and authenticating with a key or token.

Run the following cell to define the endpoint. Note that the name of the endpoint has to be unique. You'll use the `datetime` function to generate a unique name.

In [3]:
from azure.ai.ml.entities import ManagedOnlineEndpoint
import datetime

# online_endpoint_name = "endpoint-" + datetime.datetime.now().strftime("%m%d%H%M%f")
online_endpoint_name = "endpoint-11241440997965"

# create an online endpoint
endpoint = ManagedOnlineEndpoint(
    name=online_endpoint_name,
    description="Online endpoint for MLflow diabetes model",
    auth_mode="key",
)

Next, you'll create the endpoint by running the following cell. This may take several minutes. While your endpoint is being created, you can read about [what are Azure Machine Learning endpoints](https://learn.microsoft.com/azure/machine-learning/concept-endpoints).

In [3]:
ml_client.begin_create_or_update(endpoint).result()

ManagedOnlineEndpoint({'public_network_access': 'Enabled', 'provisioning_state': 'Succeeded', 'scoring_uri': 'https://endpoint-11241440997965.westeurope.inference.ml.azure.com/score', 'openapi_uri': 'https://endpoint-11241440997965.westeurope.inference.ml.azure.com/swagger.json', 'name': 'endpoint-11241440997965', 'description': 'Online endpoint for MLflow diabetes model', 'tags': {}, 'properties': {'createdBy': 'Dominik Mika', 'createdAt': '2025-11-24T14:40:50.562938+0000', 'lastModifiedAt': '2025-11-24T14:40:50.562938+0000', 'azureml.onlineendpointid': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandaidevml-rg/providers/microsoft.machinelearningservices/workspaces/polandaidevml-mlw/onlineendpoints/endpoint-11241440997965', 'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/providers/Microsoft.MachineLearningServices/locations/westeurope/mfeOperationsStatus/oeidp:98a4e2ed-9e4a-44b6-a65e-e3273c9dc09b:5b96

<p style="color:red;font-size:120%;background-color:yellow;font-weight:bold"> IMPORTANT! Wait until the endpoint is created successfully before continuing! A green notification should appear in the studio. </p>

## Configure the deployment

You can deploy multiple models to an endpoint. This is mostly useful when you want to update the deployed model while keeping the current model in production. You'll need to configure the deployment to specify which model needs to be deployed to an endpoint. In the following cell, you'll refer to the model trained and stored in the local `model` folder (stored in the same folder as this notebook). Note that since you're working with an MLflow model, you don't need to specify the environment or scoring script.

You'll also specify the infrastructure needed for the model to be deployed.

In [5]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

model = Model(
    path="./model",
    type=AssetTypes.MLFLOW_MODEL,
    name="diabetes_deployment_model",
    description="my sample mlflow model",
)
ml_client.models.create_or_update(model)

Model({'job_name': None, 'intellectual_property': None, 'system_metadata': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'diabetes_deployment_model', 'description': 'my sample mlflow model', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/models/diabetes_deployment_model/versions/1', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn/azure-labs/azure-ml-dev/Labs/11', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x721115bce9e0>, 'serialize': <msrest.serialization.Serializer object at 0x721115bcf1c0>, 'version': '1', 'latest_version': None, 'path': 'azureml://subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/w

In [6]:
from azure.ai.ml.entities import  ManagedOnlineDeployment

# create a blue deployment
model = ml_client.models.get(name="diabetes_deployment_model", version=1)
blue_deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=online_endpoint_name,
    model=model,
    instance_type="Standard_D2as_v4",
    instance_count=1,
)

## Create the deployment

Finally, you can actually deploy the model to the endpoint by running the following cell:

In [7]:
ml_client.online_deployments.begin_create_or_update(blue_deployment).result()

Check: endpoint endpoint-11241440997965 exists


..............................................................

ManagedOnlineDeployment({'private_network_connection': None, 'package_model': False, 'provisioning_state': 'Succeeded', 'endpoint_name': 'endpoint-11241440997965', 'type': 'Managed', 'name': 'blue', 'description': None, 'tags': {}, 'properties': {'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/providers/Microsoft.MachineLearningServices/locations/westeurope/mfeOperationsStatus/odidp:98a4e2ed-9e4a-44b6-a65e-e3273c9dc09b:f3b65a7a-d6df-4cab-96fb-06c35be7eeb7?api-version=2023-04-01-preview'}, 'print_as_yaml': False, 'id': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/onlineEndpoints/endpoint-11241440997965/deployments/blue', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn/azure-labs/azure-ml-dev/Labs/11', 'creation_context

The deployment of the model may take 10-15 minutes. While waiting for the model to be deployed, you can learn more about [managed endpoints in this video](https://www.youtube.com/watch?v=SxFGw_OBxNM&ab_channel=MicrosoftDeveloper).

<p style="color:red;font-size:120%;background-color:yellow;font-weight:bold"> IMPORTANT! Wait until the deployment is completed before continuing! A green notification should appear in the studio.</p>

Since you only have one model deployed to the endpoint, you want this deployment to take 100% of the traffic. If you deploy multiple models to the endpoint, you could use the same approach to distribute traffic across the deployed models.

In [20]:
# blue deployment takes 100 traffic
endpoint.traffic = {"blue": 100}
ml_client.begin_create_or_update(endpoint).result()

Readonly attribute principal_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>
Readonly attribute tenant_id will be ignored in class <class 'azure.ai.ml._restclient.v2022_05_01.models._models_py3.ManagedServiceIdentity'>


ManagedOnlineEndpoint({'public_network_access': 'Enabled', 'provisioning_state': 'Succeeded', 'scoring_uri': 'https://endpoint-11241440997965.westeurope.inference.ml.azure.com/score', 'openapi_uri': 'https://endpoint-11241440997965.westeurope.inference.ml.azure.com/swagger.json', 'name': 'endpoint-11241440997965', 'description': 'Online endpoint for MLflow diabetes model', 'tags': {}, 'properties': {'createdBy': 'Dominik Mika', 'createdAt': '2025-11-24T14:40:50.562938+0000', 'lastModifiedAt': '2025-11-25T09:23:38.031174+0000', 'azureml.onlineendpointid': '/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourcegroups/polandaidevml-rg/providers/microsoft.machinelearningservices/workspaces/polandaidevml-mlw/onlineendpoints/endpoint-11241440997965', 'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/providers/Microsoft.MachineLearningServices/locations/westeurope/mfeOperationsStatus/oeidp:98a4e2ed-9e4a-44b6-a65e-e3273c9dc09b:b9c8

<p style="color:red;font-size:120%;background-color:yellow;font-weight:bold"> IMPORTANT! Wait until the blue deployment is configured before continuing! A green notification should appear in the studio. </p> 

## Test the deployment

Let's test the deployed model by invoking the endpoint. A JSON file with sample data is used as input. The trained model predicts whether a patient has diabetes or not, based on medical data like age, BMI, and the number of pregnancies. A `[0]` indicates a patient doesn't have diabetes. A `[1]` means a patient does have diabetes.

In [8]:
# test the blue deployment with some sample data
response = ml_client.online_endpoints.invoke(
    endpoint_name=online_endpoint_name,
    deployment_name="blue",
    request_file="sample-data.json",
)

if response[1]=='1':
    print("Diabetic")
else:
    print ("Not diabetic")

Diabetic


Optionally, you can change the values in the `sample-data.json` file to try and get a different prediction.

## List endpoints

Although you can view all endpoints in the Studio, you can also list all endpoints using the SDK:

In [9]:
endpoints = ml_client.online_endpoints.list()
for endp in endpoints:
    print(endp.name)

endpoint-11241440997965


## Get endpoint details

If you want more information about a specific endpoint, you can explore the details using the SDK too.

In [10]:
# Get the details for online endpoint
endpoint = ml_client.online_endpoints.get(name=online_endpoint_name)

# existing traffic details
print(endpoint.traffic)

# Get the scoring URI
print(endpoint.scoring_uri)

{'blue': 100}
https://endpoint-11241440997965.westeurope.inference.ml.azure.com/score


## Delete the endpoint and deployment

As an endpoint is always available, it can't be paused to save costs. To avoid unnecessary costs, delete the endpoint.

In [ ]:
ml_client.online_endpoints.begin_delete(name=online_endpoint_name)

## Use Custom Scoring Script

### Test 

In [ ]:
from pathlib import Path
import mlflow.pyfunc
# 1. Get the model metadata
model = ml_client.models.get(name="mlflow-diabetes", version=1)

# 2. Download locally
download_path = Path("./aml-model")
download_path.mkdir(exist_ok=True)
ml_client.models.download(
    name=model.name,
    version=model.version,
    download_path=str(download_path)
)

# 3. Load the MLflow model
loaded_model = mlflow.pyfunc.load_model(download_path / "mlflow-diabetes/model")

# # 4. Create sample input (pandas DataFrame expected by mlflow.pyfunc)
# sample = pd.DataFrame([[1.2, 3.4, 5.6, 7.8]], 
#                       columns=["feat1", "feat2", "feat3", "feat4"])

# # 5. Predict
# pred = loaded_model.predict(sample)
# pred
